# BABA (Alibaba) — EDA + LSTM baseline

Exploratory analysis of BABA closing price + naive baseline + small LSTM prototype.
Final training is in `src/model/train.py`; this notebook informs hyperparameters.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import DATA, MODEL, RAW_PARQUET
from src.data.fetch import fetch
from src.data.preprocess import build_splits
from src.model.architecture import build_lstm
from src.model.evaluate import all_metrics

## 1. Load data

In [ ]:
if not RAW_PARQUET.exists():
    fetch()
df = pd.read_parquet(RAW_PARQUET).set_index('Date').sort_index()
print(df.shape)
df.head()

## 2. Price + volume

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
df['Close'].plot(ax=ax1, title=f'{DATA.symbol} Close')
ax1.set_ylabel('USD')
df['Volume'].plot(ax=ax2, color='gray')
ax2.set_ylabel('Volume')
plt.tight_layout()
plt.show()

## 3. Stationarity quick-check (returns vs price)
Stock prices are non-stationary (random walk-ish); daily returns are closer to stationary.

In [ ]:
rets = df['Close'].pct_change().dropna()
fig, ax = plt.subplots(1, 2, figsize=(12, 3))
df['Close'].rolling(60).mean().plot(ax=ax[0], label='Close (60d MA)')
df['Close'].plot(ax=ax[0], alpha=0.4)
ax[0].legend(); ax[0].set_title('Close + 60-day MA')
rets.plot(ax=ax[1], title='Daily returns')
plt.tight_layout(); plt.show()
print('Mean return', rets.mean(), 'Std', rets.std())

## 4. Naive baseline — predict yesterday's close
Sanity floor: any model that doesn't beat this is doing nothing useful.

In [ ]:
close = df['Close'].values
y_true = close[1:]
y_naive = close[:-1]
print('Naive (y_hat = y_{t-1}) on full series:', all_metrics(y_true, y_naive))

## 5. Small LSTM prototype
Few epochs, used to validate the pipeline before launching the full `python -m src.model.train` run.

In [ ]:
splits = build_splits(df=df.reset_index(), lookback=MODEL.lookback)
model = build_lstm(units=32)
model.fit(splits.X_train, splits.y_train,
          validation_data=(splits.X_val, splits.y_val),
          epochs=5, batch_size=32, verbose=2);

In [ ]:
y_pred_s = model.predict(splits.X_test, verbose=0).ravel()
y_pred = splits.scaler.inverse_transform(y_pred_s.reshape(-1, 1)).ravel()
y_true = splits.scaler.inverse_transform(splits.y_test.reshape(-1, 1)).ravel()
print('LSTM prototype (test):', all_metrics(y_true, y_pred))

plt.figure(figsize=(12, 4))
plt.plot(y_true, label='actual')
plt.plot(y_pred, label='predicted', alpha=0.8)
plt.legend(); plt.title(f'{DATA.symbol} — test split: actual vs predicted'); plt.show()

## Notes for production training
- Start with `lookback=60`, two LSTM(50)+Dropout(0.2) layers (see `src/model/architecture.py`).
- `EarlyStopping(patience=10, restore_best_weights=True)` keeps the best epoch.
- Target MAPE ~3-5%; that's competitive with naive but on a non-stationary series so report alongside the naive baseline.
- Future work: include volume + macro features (multivariate) and/or predict returns instead of price.